In [1]:
import pandas as pd
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory

In [2]:
esferas =['A', 'B']
metais = ['Ferro', 'Chumbo','Estanho']

requisitos = {
    'Ferro': 16,
    'Chumbo': 11,
    'Estanho': 15
}

informações ={
    'A': {'Ferro': 2, 'Chumbo': 1, 'Estanho': 1, 'Lucro': 300},
    'B': {'Ferro': 1, 'Chumbo': 2, 'Estanho': 3, 'Lucro': 800}
}

In [8]:
model = pyo.ConcreteModel()

model.esferas = pyo.Set(initialize=esferas)
model.metais = pyo.Set(initialize=metais)
model.x = pyo.Var(model.esferas,bounds=(0, None), domain=pyo.NonNegativeIntegers)

def objetivo(model):
    return sum(
        model.x[e] * informações[e]['Lucro'] for e in model.esferas
    )
model.objetivo = pyo.Objective(rule=objetivo, sense=pyo.maximize)

def restrições(model, m):
    return sum(
        model.x[e] * informações[e][m] for e in model.esferas
    ) <= requisitos[m]
model.restrições = pyo.Constraint(model.metais, rule=restrições)

In [9]:
model.pprint()

2 Set Declarations
    esferas : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    2 : {'A', 'B'}
    metais : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    3 : {'Ferro', 'Chumbo', 'Estanho'}

1 Var Declarations
    x : Size=2, Index=esferas
        Key : Lower : Value : Upper : Fixed : Stale : Domain
          A :     0 :  None :  None : False :  True : NonNegativeIntegers
          B :     0 :  None :  None : False :  True : NonNegativeIntegers

1 Objective Declarations
    objetivo : Size=1, Index=None, Active=True
        Key  : Active : Sense    : Expression
        None :   True : maximize : 300*x[A] + 800*x[B]

1 Constraint Declarations
    restrições : Size=3, Index=metais, Active=True
        Key     : Lower : Body          : Upper : Active
         Chumbo :  -Inf : x[A] + 2*x[B] :  11.0 :   True
        Estanho :  -Inf : x[A] + 3*x[B]

In [11]:
opt = SolverFactory('cplex')
resultados = opt.solve(model, tee=True)



Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.1.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2022.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\joaon\AppData\Local\Temp\tmp327yqnwo.cplex.log' open.
CPLEX> Problem 'C:\Users\joaon\AppData\Local\Temp\tmpz5_0knss.pyomo.lp' read.
Read time = 0.02 sec. (0.00 ticks)
CPLEX> Problem name         : C:\Users\joaon\AppData\Local\Temp\tmpz5_0knss.pyomo.lp
Objective sense      : Maximize
Variables            :       2  [General Integer: 2]
Objective nonzeros   :       2
Linear constraints   :       3  [Less: 3]
  Nonzeros           :       6
  RHS nonzeros       :       3

Variables            : Min LB: 0.000000         Max UB: all infinite   
Objective nonzeros   : Min   : 300.0000       

In [14]:
model.x.pprint()

x : Size=2, Index=esferas
    Key : Lower : Value              : Upper : Fixed : Stale : Domain
      A :     0 : 2.9999999999999996 :  None : False : False : NonNegativeIntegers
      B :     0 :                4.0 :  None : False : False : NonNegativeIntegers


In [16]:
model.objetivo()

4100.0